# Day 09. Exercise 00
# Regularization

## 0. Imports

In [87]:
import pandas as pd
import numpy as np
import time
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error



## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [88]:
df = pd.read_csv("./sample_data/dayofweek.csv")
df

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.756764,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.724861,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.692958,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.661055,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,-0.533442,0.945382,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,-0.629151,0.945382,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,-0.597248,0.945382,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,-0.565345,0.945382,3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [89]:
y = df["dayofweek"]
X = df.drop("dayofweek", axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=21, test_size=0.2, stratify=df["dayofweek"])

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [90]:
%%time
def cross_valid(n_splits, X, y, model):
  skf = StratifiedKFold(n_splits=n_splits, random_state=21, shuffle=True)
  res = []
  for i, j in skf.split(X, y):
    X_train_kf, X_valid = X.iloc[i], X.iloc[j]
    y_train_kf, y_valid = y.iloc[i], y.iloc[j]
    model.fit(X_train_kf, y_train_kf)

    tr_ac = accuracy_score(y_train_kf, model.predict(X_train_kf))
    vd_ac = accuracy_score(y_valid, model.predict(X_valid))

    res = res + [vd_ac]
    print(f'train -  {tr_ac:.5}   |   valid -  {vd_ac:.5}')

  print(f"Average accuracy on crossval is {np.mean(res):.5}")
  print(f"Std is {np.std(res):.5}")
cross_valid(10, X_train, y_train, LogisticRegression(random_state=21, fit_intercept=False))

train -  0.64056   |   valid -  0.65926
train -  0.63561   |   valid -  0.62222
train -  0.64468   |   valid -  0.6
train -  0.64056   |   valid -  0.64444
train -  0.65375   |   valid -  0.60741
train -  0.62902   |   valid -  0.6
train -  0.66117   |   valid -  0.6
train -  0.63726   |   valid -  0.54074
train -  0.63756   |   valid -  0.66418
train -  0.64745   |   valid -  0.61194
Average accuracy on crossval is 0.61502
Std is 0.03399
CPU times: user 1.53 s, sys: 8.82 ms, total: 1.54 s
Wall time: 1.36 s


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [91]:
penalties = [None, "l1", "l2"]
solvers = ["saga", "liblinear", "lbfgs", "sag", "newton-cg"]

for i in penalties:
  for j in solvers:
    if 'saga' == j or ("lbfgs" == j and i != "l1") or ("liblinear" == j and i != None) or (i in ['sag', 'newton-cg'] and i != "l1"):
      start_time = time.time()
      model = LogisticRegression(random_state=21, fit_intercept=False, solver=j, penalty=i, max_iter=5000)
      print("penalty", i , "solver", j)
      cross_valid(10, X_train, y_train, model)
      end_time = time.time()
      print(f"⏱️  Время выполнения: {end_time - start_time:.2f} сек\n")
      print()
      print()

penalty None solver saga
train -  0.66035   |   valid -  0.67407
train -  0.67189   |   valid -  0.60741
train -  0.65787   |   valid -  0.62963
train -  0.66117   |   valid -  0.66667
train -  0.66777   |   valid -  0.60741
train -  0.64716   |   valid -  0.62963
train -  0.66859   |   valid -  0.59259
train -  0.66447   |   valid -  0.60741
train -  0.6598   |   valid -  0.70149
train -  0.66969   |   valid -  0.62687
Average accuracy on crossval is 0.63432
Std is 0.033395
⏱️  Время выполнения: 56.72 сек



penalty None solver lbfgs
train -  0.65952   |   valid -  0.67407
train -  0.67189   |   valid -  0.60741
train -  0.65787   |   valid -  0.62963
train -  0.66035   |   valid -  0.66667
train -  0.66777   |   valid -  0.60741
train -  0.64716   |   valid -  0.62963
train -  0.66859   |   valid -  0.59259
train -  0.66447   |   valid -  0.60741
train -  0.66063   |   valid -  0.70149
train -  0.67133   |   valid -  0.62687
Average accuracy on crossval is 0.63432
Std is 0.033395
⏱️ 

## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [92]:
%%time
cross_valid(10, X_train, y_train, SVC(probability=True, kernel="linear", random_state=21))

train -  0.70651   |   valid -  0.68148
train -  0.6892   |   valid -  0.64444
train -  0.69744   |   valid -  0.66667
train -  0.6892   |   valid -  0.65926
train -  0.69497   |   valid -  0.63704
train -  0.68673   |   valid -  0.68148
train -  0.69827   |   valid -  0.61481
train -  0.70486   |   valid -  0.57778
train -  0.68863   |   valid -  0.72388
train -  0.71005   |   valid -  0.64179
Average accuracy on crossval is 0.65286
Std is 0.038003
CPU times: user 5.98 s, sys: 11.8 ms, total: 5.99 s
Wall time: 6.01 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [93]:
%%time
cross_valid(10, X_train, y_train, SVC(probability=True, kernel="linear", random_state=21, C=0.5))

train -  0.67436   |   valid -  0.65185
train -  0.66612   |   valid -  0.62963
train -  0.67024   |   valid -  0.65926
train -  0.66777   |   valid -  0.62963
train -  0.67189   |   valid -  0.62963
train -  0.66035   |   valid -  0.62222
train -  0.68261   |   valid -  0.59259
train -  0.67766   |   valid -  0.57037
train -  0.67298   |   valid -  0.70896
train -  0.67792   |   valid -  0.60448
Average accuracy on crossval is 0.62986
Std is 0.036379
CPU times: user 4.73 s, sys: 1.16 ms, total: 4.73 s
Wall time: 4.73 s


In [94]:
%%time
cross_valid(10, X_train, y_train, SVC(probability=True, kernel="linear", random_state=21, C=1.5))

train -  0.71805   |   valid -  0.68889
train -  0.69085   |   valid -  0.64444
train -  0.70816   |   valid -  0.66667
train -  0.70074   |   valid -  0.66667
train -  0.70734   |   valid -  0.64444
train -  0.69415   |   valid -  0.68889
train -  0.71146   |   valid -  0.64444
train -  0.72053   |   valid -  0.59259
train -  0.70346   |   valid -  0.74627
train -  0.70675   |   valid -  0.65672
Average accuracy on crossval is 0.664
Std is 0.037843
CPU times: user 5.75 s, sys: 6.06 ms, total: 5.75 s
Wall time: 5.81 s


In [95]:
%%time
cross_valid(10, X_train, y_train, SVC(probability=True, kernel="linear", random_state=21, C=2))

train -  0.72795   |   valid -  0.68148
train -  0.70981   |   valid -  0.64444
train -  0.71476   |   valid -  0.67407
train -  0.7197   |   valid -  0.6963
train -  0.70981   |   valid -  0.64444
train -  0.6925   |   valid -  0.68148
train -  0.71476   |   valid -  0.64444
train -  0.73619   |   valid -  0.6
train -  0.70264   |   valid -  0.75373
train -  0.73064   |   valid -  0.67164
Average accuracy on crossval is 0.6692
Std is 0.038521
CPU times: user 5.11 s, sys: 7.04 ms, total: 5.12 s
Wall time: 5.14 s


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [96]:
%%time
cross_valid(10, X_train, y_train, DecisionTreeClassifier(max_depth=10, random_state=21))

train -  0.80874   |   valid -  0.77037
train -  0.79802   |   valid -  0.7037
train -  0.81286   |   valid -  0.72593
train -  0.80049   |   valid -  0.74815
train -  0.80956   |   valid -  0.68889
train -  0.78978   |   valid -  0.74074
train -  0.80627   |   valid -  0.60741
train -  0.82688   |   valid -  0.71111
train -  0.78995   |   valid -  0.79104
train -  0.80313   |   valid -  0.70896
Average accuracy on crossval is 0.71963
Std is 0.047909
CPU times: user 107 ms, sys: 1.74 ms, total: 108 ms
Wall time: 108 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [97]:
%%time
cross_valid(10, X_train, y_train, DecisionTreeClassifier(max_depth=5, random_state=21, min_samples_split=10))

train -  0.58285   |   valid -  0.60741
train -  0.57543   |   valid -  0.52593
train -  0.61253   |   valid -  0.6
train -  0.58862   |   valid -  0.58519
train -  0.5845   |   valid -  0.5037
train -  0.56142   |   valid -  0.53333
train -  0.5812   |   valid -  0.51852
train -  0.62407   |   valid -  0.51111
train -  0.57414   |   valid -  0.56716
train -  0.5659   |   valid -  0.48507
Average accuracy on crossval is 0.54374
Std is 0.040812
CPU times: user 115 ms, sys: 1.01 ms, total: 116 ms
Wall time: 117 ms


In [98]:
%%time
cross_valid(10, X_train, y_train, DecisionTreeClassifier(max_depth=5, random_state=21, min_samples_split=15))

train -  0.57873   |   valid -  0.61481
train -  0.57378   |   valid -  0.53333
train -  0.61088   |   valid -  0.6
train -  0.58368   |   valid -  0.58519
train -  0.58285   |   valid -  0.5037
train -  0.56059   |   valid -  0.52593
train -  0.57708   |   valid -  0.51111
train -  0.62242   |   valid -  0.51111
train -  0.57249   |   valid -  0.56716
train -  0.56178   |   valid -  0.48507
Average accuracy on crossval is 0.54374
Std is 0.042524
CPU times: user 97.8 ms, sys: 1.99 ms, total: 99.8 ms
Wall time: 101 ms


In [99]:
%%time
cross_valid(10, X_train, y_train, DecisionTreeClassifier(max_depth=10, random_state=21, min_samples_split=15))

train -  0.76917   |   valid -  0.74815
train -  0.7601   |   valid -  0.71852
train -  0.76834   |   valid -  0.68889
train -  0.7601   |   valid -  0.71111
train -  0.77824   |   valid -  0.67407
train -  0.75433   |   valid -  0.71852
train -  0.77411   |   valid -  0.6
train -  0.78071   |   valid -  0.66667
train -  0.76277   |   valid -  0.77612
train -  0.76359   |   valid -  0.6791
Average accuracy on crossval is 0.69811
Std is 0.046076
CPU times: user 119 ms, sys: 985 µs, total: 120 ms
Wall time: 121 ms


In [100]:
%%time
cross_valid(10, X_train, y_train, DecisionTreeClassifier(max_depth=15, random_state=21, min_samples_split=5))

train -  0.93982   |   valid -  0.85926
train -  0.92993   |   valid -  0.82222
train -  0.93899   |   valid -  0.86667
train -  0.9291   |   valid -  0.8963
train -  0.9357   |   valid -  0.87407
train -  0.92086   |   valid -  0.8
train -  0.92086   |   valid -  0.85926
train -  0.95136   |   valid -  0.85926
train -  0.92422   |   valid -  0.8806
train -  0.9341   |   valid -  0.80597
Average accuracy on crossval is 0.85236
Std is 0.030575
CPU times: user 128 ms, sys: 0 ns, total: 128 ms
Wall time: 129 ms


In [101]:
%%time
cross_valid(10, X_train, y_train, DecisionTreeClassifier(max_depth=25, random_state=21, min_samples_split=20))

train -  0.86068   |   valid -  0.79259
train -  0.87222   |   valid -  0.8
train -  0.86645   |   valid -  0.82222
train -  0.87552   |   valid -  0.83704
train -  0.85408   |   valid -  0.74074
train -  0.87552   |   valid -  0.74074
train -  0.87222   |   valid -  0.73333
train -  0.88293   |   valid -  0.79259
train -  0.88386   |   valid -  0.86567
train -  0.86573   |   valid -  0.77612
Average accuracy on crossval is 0.79011
Std is 0.04165
CPU times: user 121 ms, sys: 3.96 ms, total: 125 ms
Wall time: 127 ms


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [102]:
%%time
cross_valid(10, X_train, y_train, RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21))

train -  0.97939   |   valid -  0.85185
train -  0.9662   |   valid -  0.85926
train -  0.96208   |   valid -  0.91852
train -  0.97115   |   valid -  0.91852
train -  0.97197   |   valid -  0.88148
train -  0.96538   |   valid -  0.86667
train -  0.96455   |   valid -  0.88889
train -  0.96867   |   valid -  0.87407
train -  0.96458   |   valid -  0.93284
train -  0.96787   |   valid -  0.86567
Average accuracy on crossval is 0.88578
Std is 0.026734
CPU times: user 1.49 s, sys: 4.03 ms, total: 1.49 s
Wall time: 1.49 s


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [103]:
%%time
cross_valid(10, X_train, y_train, RandomForestClassifier(n_estimators=50, max_depth=5, random_state=21, min_samples_leaf = 10))

train -  0.56554   |   valid -  0.57778
train -  0.5235   |   valid -  0.52593
train -  0.51855   |   valid -  0.52593
train -  0.53504   |   valid -  0.6
train -  0.57131   |   valid -  0.55556
train -  0.52102   |   valid -  0.48889
train -  0.53998   |   valid -  0.5037
train -  0.54905   |   valid -  0.51852
train -  0.51236   |   valid -  0.52239
train -  0.54036   |   valid -  0.47015
Average accuracy on crossval is 0.52888
Std is 0.03743
CPU times: user 1.14 s, sys: 3.01 ms, total: 1.15 s
Wall time: 1.15 s


In [104]:
%%time
cross_valid(10, X_train, y_train, RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21, min_samples_leaf = 15))

train -  0.60758   |   valid -  0.62963
train -  0.60676   |   valid -  0.58519
train -  0.60016   |   valid -  0.62963
train -  0.58697   |   valid -  0.64444
train -  0.60758   |   valid -  0.53333
train -  0.58697   |   valid -  0.53333
train -  0.61253   |   valid -  0.56296
train -  0.60676   |   valid -  0.53333
train -  0.59473   |   valid -  0.6194
train -  0.62685   |   valid -  0.53731
Average accuracy on crossval is 0.58086
Std is 0.043929
CPU times: user 1.23 s, sys: 4.04 ms, total: 1.23 s
Wall time: 1.23 s


In [105]:
%%time
cross_valid(10, X_train, y_train, RandomForestClassifier(n_estimators=50, max_depth=15, random_state=21, min_samples_leaf = 5))

train -  0.78483   |   valid -  0.77037
train -  0.7939   |   valid -  0.77037
train -  0.77988   |   valid -  0.74074
train -  0.79308   |   valid -  0.77778
train -  0.78895   |   valid -  0.6963
train -  0.78071   |   valid -  0.76296
train -  0.78236   |   valid -  0.71111
train -  0.77411   |   valid -  0.66667
train -  0.7883   |   valid -  0.80597
train -  0.80148   |   valid -  0.6791
Average accuracy on crossval is 0.73814
Std is 0.044663
CPU times: user 1.76 s, sys: 7.91 ms, total: 1.77 s
Wall time: 1.81 s


In [106]:
%%time
cross_valid(10, X_train, y_train, RandomForestClassifier(n_estimators=50, max_depth=10, random_state=21, min_samples_leaf = 15))

train -  0.59275   |   valid -  0.6
train -  0.58697   |   valid -  0.55556
train -  0.57543   |   valid -  0.57037
train -  0.58862   |   valid -  0.64444
train -  0.60841   |   valid -  0.54074
train -  0.58533   |   valid -  0.55556
train -  0.61748   |   valid -  0.55556
train -  0.5911   |   valid -  0.51111
train -  0.58979   |   valid -  0.6194
train -  0.61862   |   valid -  0.53731
Average accuracy on crossval is 0.569
Std is 0.038589
CPU times: user 1.78 s, sys: 3.8 ms, total: 1.78 s
Wall time: 1.85 s


In [107]:
%%time
cross_valid(10, X_train, y_train, RandomForestClassifier(n_estimators=50, max_depth=20, random_state=21, min_samples_leaf = 5))

train -  0.79637   |   valid -  0.77778
train -  0.80049   |   valid -  0.74815
train -  0.78895   |   valid -  0.73333
train -  0.80049   |   valid -  0.77037
train -  0.80956   |   valid -  0.71111
train -  0.80791   |   valid -  0.77037
train -  0.78483   |   valid -  0.6963
train -  0.78648   |   valid -  0.68889
train -  0.80231   |   valid -  0.81343
train -  0.80066   |   valid -  0.70149
Average accuracy on crossval is 0.74112
Std is 0.0395
CPU times: user 1.32 s, sys: 1.03 ms, total: 1.32 s
Wall time: 1.32 s


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [108]:
rf = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.908284023668639

In [109]:
errors = y_test != y_pred
data = pd.DataFrame({"day": y_test, "errors": errors})
data.groupby("day").agg({"errors": "sum"})

,errors
day,
0,7
1,7
2,2
3,1
4,4
5,3
6,7


In [110]:
joblib.dump(rf, "model.pkl")

['model.pkl']